# Backward Chaining — Identify the Animal

A **goal-driven** _(a.k.a. *top-down*)_ production-rule engine, using the exact same rule base as the forward-chaining version of this example, but a completely different traversal strategy.

**Strategy.** Start from a single **goal** _(e.g., "is this animal a cheetah?")_ and work *backwards*:

1. If a rule concludes the goal, recursively try to prove **all** of that rule's premises _(each premise becomes a new sub-goal)_.
1. If several rules could conclude the same goal, try them **in order**, the first one whose premises all succeed proves the goal; if one fails, **backtrack** and try the next candidate rule.
1. If *no* rule concludes a predicate, it's a **primitive / askable fact**, ask the user _(or consult a scripted answer)_ rather than search further.

This only ever explores rules that are actually relevant to the goal. Unlike forward chaining, it never wastes work deriving facts nobody asked about.

---

In [1]:
versioninfo()  # -> v"1.11.7"

Julia Version 1.11.7
Commit f2b3dbda30a (2025-09-08 12:10 UTC)
Build Info:
  Official https://julialang.org/ release
Platform Info:
  OS: Linux (x86_64-linux-gnu)
  CPU: 8 × Intel(R) Core(TM) i7-8565U CPU @ 1.80GHz
  WORD_SIZE: 64
  LLVM: libLLVM-16.0.6 (ORCJIT, skylake)
Threads: 1 default, 0 interactive, 1 GC (on 8 virtual cores)
Environment:
  JULIA_GPG = 3673DF529D9049477F76B37566E3C7DC03D6E495
  JULIA_PATH = /usr/local/julia
  JULIA_DEPOT_PATH = /root/.julia
  JULIA_VERSION = 1.11.7


---

## Data Structure

Identical `Rule` representation to the forward-chaining engine: a name (for tracing), a conjunction of `premises`, and one `conclusion`. Same propositional simplification applies here top, one ground `Symbol` per fact, implicitly about a single individual.

In [2]:
module Chaining

    export ¬, Rule
    export rules_by_conclusion, ask_user
    export backward_chain!
    
    include("Chaining.jl")

end

Main.Chaining

In [3]:
using .Chaining

## The Rule Base

Same rules, same order, as the forward-chaining version: 

**Backward chaining is order-sensitive**. When several rules conclude the same predicate _($\mathfrak{R}_5$ and $\mathfrak{R}_6$ both conclude `:is_carnivore`)_, they're tried *in the order listed*, so $\mathfrak{R}_5$ is always attempted before $\mathfrak{R}_6$. Forward chaining doesn't care about rule order because it eventually fires every satisfiable rule anyway; backward chaining stops at the *first* rule that succeeds, so order can change which questions get asked _(though not, for a correct/consistent rule base, the final true/false verdict)_.

In [4]:
# NOTE: order matters here, when several rules conclude the same predicate, they are tried in the order listed (R5 before R6).
const RULES = Rule[
    Rule("R1",  [:has_hair], :is_mammal),
    Rule("R2",  [:gives_milk], :is_mammal),
    Rule("R3",  [:has_feathers], :is_bird),
    Rule("R4",  [:flies, :lays_eggs], :is_bird),
    Rule("R5",  [:is_mammal, :eats_meat], :is_carnivore),
    Rule("R6",  [:is_mammal, :has_pointed_teeth, :has_claws, :has_forward_eyes], :is_carnivore),
    Rule("R7",  [:is_carnivore, :has_tawny_color, :has_dark_spots], :is_cheetah),
    Rule("R8",  [:is_carnivore, :has_tawny_color, :has_black_stripes], :is_tiger),
    Rule("R9",  [:is_bird, ¬(:flies), :has_long_neck, :has_long_legs], :is_ostrich),
    Rule("R10", [:is_bird, ¬(:flies), :swims, :has_black_white_color], :is_penguin),
    Rule("R11", [:is_bird, :is_good_flyer], :is_albatross)
]

11-element Vector{Rule}:
 Rule("R1", Union{Main.Chaining.Neg, Symbol}[:has_hair], :is_mammal)
 Rule("R2", Union{Main.Chaining.Neg, Symbol}[:gives_milk], :is_mammal)
 Rule("R3", Union{Main.Chaining.Neg, Symbol}[:has_feathers], :is_bird)
 Rule("R4", Union{Main.Chaining.Neg, Symbol}[:flies, :lays_eggs], :is_bird)
 Rule("R5", Union{Main.Chaining.Neg, Symbol}[:is_mammal, :eats_meat], :is_carnivore)
 Rule("R6", Union{Main.Chaining.Neg, Symbol}[:is_mammal, :has_pointed_teeth, :has_claws, :has_forward_eyes], :is_carnivore)
 Rule("R7", Union{Main.Chaining.Neg, Symbol}[:is_carnivore, :has_tawny_color, :has_dark_spots], :is_cheetah)
 Rule("R8", Union{Main.Chaining.Neg, Symbol}[:is_carnivore, :has_tawny_color, :has_black_stripes], :is_tiger)
 Rule("R9", Union{Main.Chaining.Neg, Symbol}[:is_bird, Main.Chaining.Neg(:flies), :has_long_neck, :has_long_legs], :is_ostrich)
 Rule("R10", Union{Main.Chaining.Neg, Symbol}[:is_bird, Main.Chaining.Neg(:flies), :swims, :has_black_white_color], :is_penguin)
 Ru

## The backward-chaining engine

## Demo

Some facts are pre-seeded into `known_true` _(as if already observed)_, and the rest are answered via a **scripted** `answers` dict so the run is
reproducible without stdin input. 

1. Goal `is_cheetah` → only $\mathfrak{R}_7$ concludes it → needs `is_carnivore` (first premise), then `has_tawny_color`, `has_dark_spots`.
1. `is_carnivore` → tries `R5` first → needs `is_mammal` (proved via $\mathfrak{R}_1$ from the already-known `has_hair`) and `eats_meat` → **asked, scripted    `false`** → $\mathfrak{R}_5$ fails → **backtrack**.
1. Tries $\mathfrak{R}_5$ → needs `is_mammal` (cached from step 2), `has_pointed_teeth`, `has_claws`, `has_forward_eyes` → all **asked, scripted `true`** → $\mathfrak{R}_5$(4 4 succeeds → `is_carnivore = true`.
1. Back in $\mathfrak{R}_5$: `has_tawny_color` and `has_dark_spots` are already in `known_true` → all premises hold → `is_cheetah = true`.

So only **4 questions** get asked total _(`eats_meat`, `has_pointed_teeth`, `has_claws`, `has_forward_eyes`)_, nothing about feathers, swimming, or stripes, since those predicates are never on the path from `is_cheetah` back to a primitive fact. That's the practical payoff of goal-driven search: it only asks what's relevant to the goal at hand.

In [5]:
function main()
    rules_idx = rules_by_conclusion(RULES)

    # Facts already in working memory (never asked).
    known_true  = Set{Symbol}([:has_hair, :has_tawny_color, :has_dark_spots])
    known_false = Set{Symbol}()

    # Scripted answers reproducing the exercise: eats_meat is answered
    # "unknown" (-> false, for proof purposes), the other three "yes".
    answers = Dict{Symbol,Bool}(
        :eats_meat          => false,
        :has_pointed_teeth  => true,
        :has_claws          => true,
        :has_forward_eyes   => true,
    )

    println("Goal: is_cheetah(a) ?\n")
    result = backward_chain!(:is_cheetah, rules_idx, known_true, known_false, answers)

    println()
    println(result ? "is_cheetah(a) = TRUE" : "is_cheetah(a) = FALSE (could not be proven)")
    println("known_true  = ", known_true)
    println("known_false = ", known_false)
end

main (generic function with 1 method)

In [6]:
main()

Goal: is_cheetah(a) ?

Goal is_cheetah: trying R7  [premises: is_carnivore, has_tawny_color, has_dark_spots]


LoadError: StackOverflowError:

## Forward vs. backward chaining _(same knowledge, different search)_

This file shares its `Rule` type and its entire `RULES` knowledge base with the forward-chaining notebook (`forward_chaining.ipynb`), same Horn clauses, same facts, and same target conclusions. What differs is purely the **traversal strategy**:

| | Forward chaining | Backward chaining |
|---|---|---|
| Direction | data → conclusions | goal → required data |
| Rule scan | *every* rule, every pass | only rules relevant to the current (sub-)goal |
| Best suited to | "what can I conclude from what I know?" (no fixed goal) | "is this specific goal true?" (goal known up front) |
| Handles multiple simultaneous conclusions | naturally (fires everything) | needs one `prove!` call per goal |
| Rule order | irrelevant to the final result | can affect *which questions get asked* (not the final true/false, for a consistent rule base) |
| Unknowns | all facts must be given up front | facts are asked **on demand**, only if actually needed |